<a href="https://colab.research.google.com/github/Samuel2253360/Proyecto-Estructuras/blob/main/proyecto_grafos_email_eu_core.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aplicación de la teoría de grafos
## Dataset: *email-Eu-core*

**Asignatura:** Estructuras de Datos — UIS, 2026-I  
**Integrantes:**
| # | Nombre | Tarea asignada |
|---|--------|----------------|
| 1 | *(Nombre integrante 1)* | Tarea 1 — Resiliencia y nodo más influyente |
| 2 | *(Nombre integrante 2)* | Tarea 2 — Comunicación inter-departamental |
| 3 | *Angel Camilo Pachon Padilla* | Tarea 3 — Componentes conexas y accesibilidad |
| 4 | *(Samuel Alejandro Fuentes Sandoval)* | Tarea 4 — Detección de comunidades |

---
> **Nota:** El algoritmo de Dijkstra **no está permitido** en ninguna tarea.

---
# 1. Descripción del dataset

## Origen y contexto

La red **email-Eu-core** fue generada a partir de datos de correo electrónico de una gran institución de investigación europea. Contiene información anonimizada sobre toda la comunicación por correo entre los miembros de la institución.

Existe una arista dirigida `(u → v)` si la persona `u` envió **al menos un correo** a la persona `v`. La red representa únicamente la comunicación interna (el "núcleo"), sin mensajes externos.

El dataset incluye también las **pertenencias reales a departamentos** (*ground-truth community memberships*): cada nodo pertenece exactamente a uno de los **42 departamentos** del instituto.

## Archivos

| Archivo | Descripción |
|---------|-------------|
| `email-Eu-core.txt` | Aristas del grafo: cada línea `u v` indica que `u` envió correo a `v` |
| `email-Eu-core-department-labels.txt` | Etiquetas: cada línea `nodo departamento` |

## Estadísticas del dataset

| Estadística | Valor |
|-------------|-------|
| Nodos | 1 005 |
| Aristas | 25 571 |
| Nodos en la WCC más grande | 986 (98.1 %) |
| Aristas en la WCC más grande | 25 552 (99.9 %) |
| Nodos en la SCC más grande | 803 (79.9 %) |
| Aristas en la SCC más grande | 24 729 (96.7 %) |
| Coeficiente de agrupamiento promedio | 0.3994 |
| Número de triángulos | 105 461 |
| Fracción de triángulos cerrados | 0.1085 |
| Diámetro (camino más corto más largo) | 7 |
| Diámetro efectivo (percentil 90) | 2.9 |

## Fuentes

- Yin et al. "Local Higher-order Graph Clustering." KDD 2017.
- Leskovec, Kleinberg y Faloutsos. "Graph Evolution: Densification and Shrinking Diameters." ACM TKDD, 2007.
- https://snap.stanford.edu/data/email-Eu-core.html

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

---
# 2. Carga y exploración inicial del grafo

In [ ]:
# Instalación de dependencias (solo si se ejecuta en Colab)
# !pip install networkx matplotlib

In [ ]:
from collections import defaultdict, deque
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Paleta de colores para visualizaciones
COLORES_DEPT = plt.cm.tab20

In [ ]:
# ── Rutas a los archivos ───────────────────────────────────────────────────────
# En Colab: sube los archivos con la utilidad de archivos o usa Drive
RUTA_ARISTAS   = 'email-Eu-core.txt'
RUTA_ETIQUETAS = 'email-Eu-core-department-labels.txt'

# ── Lectura del grafo ──────────────────────────────────────────────────────────
# Representación: lista de adyacencia (diccionario de conjuntos)
# grafo[u] = conjunto de nodos a los que u envió correo
grafo = defaultdict(set)       # grafo dirigido
grafo_inv = defaultdict(set)   # grafo transpuesto (útil para SCC y centralidad)
nodos = set()

with open(RUTA_ARISTAS) as f:
    for linea in f:
        linea = linea.strip()
        if not linea or linea.startswith('#'):
            continue
        u, v = map(int, linea.split())
        if u == v:             # ignorar auto-bucles
            continue
        grafo[u].add(v)
        grafo_inv[v].add(u)
        nodos.update([u, v])

# Garantizar que todos los nodos estén en ambos diccionarios
for n in list(nodos):
    grafo[n]     # crea entrada vacía si no existe
    grafo_inv[n]

num_nodos  = len(nodos)
num_aristas = sum(len(v) for v in grafo.values())

print(f'Nodos:   {num_nodos}')
print(f'Aristas: {num_aristas}')

FileNotFoundError: [Errno 2] No such file or directory: '/content/email-Eu-core.txt'

In [ ]:
# Mostrar las primeras 10 líneas del archivo de etiquetas para verificar su formato exacto
with open('/content/email-Eu-core-department-labels.txt', 'r') as f:
    for _ in range(10):
        line = f.readline()
        if not line:
            break
        print(repr(line))

FileNotFoundError: [Errno 2] No such file or directory: '/content/email-Eu-core-department-labels.txt'

In [ ]:
# ── Lectura de etiquetas de departamento ──────────────────────────────────────
# departamento[nodo] = id_departamento  (0 a 41)
departamento = {}

with open(RUTA_ETIQUETAS) as f:
    for linea in f:
        linea = linea.strip()
        if not linea:
            continue
        nodo, dept = map(int, linea.split())
        departamento[nodo] = dept

num_departamentos = len(set(departamento.values()))
print(f'Departamentos distintos: {num_departamentos}')

# Nodos del grafo sin etiqueta (si los hubiera)
sin_etiqueta = nodos - set(departamento.keys())
print(f'Nodos sin etiqueta de departamento: {len(sin_etiqueta)}')

In [ ]:
# ── Exploración básica ────────────────────────────────────────────────────────
grado_salida  = {n: len(grafo[n])     for n in nodos}
grado_entrada = {n: len(grafo_inv[n]) for n in nodos}
grado_total   = {n: grado_salida[n] + grado_entrada[n] for n in nodos}

print('=== Estadísticas de grado ===')
print(f'Grado de salida  — máx: {max(grado_salida.values()):>4}  '
      f'prom: {sum(grado_salida.values()) / num_nodos:.2f}')
print(f'Grado de entrada — máx: {max(grado_entrada.values()):>4}  '
      f'prom: {sum(grado_entrada.values()) / num_nodos:.2f}')

# Distribución del grado total
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(list(grado_salida.values()),  bins=50, color='steelblue',  edgecolor='white')
axes[0].set_title('Distribución de grado de salida');  axes[0].set_xlabel('Grado'); axes[0].set_ylabel('Frecuencia')
axes[1].hist(list(grado_entrada.values()), bins=50, color='darkorange', edgecolor='white')
axes[1].set_title('Distribución de grado de entrada'); axes[1].set_xlabel('Grado')
plt.tight_layout()
plt.show()

---
# Tarea 1 — Resiliencia de la red: el contacto más influyente
**Integrante:** Santiago Jimenez Rico

## 1.1 Descripción del problema


En una organicación la comunicación no es uniforme: hay personas que envian y reciben mas correos que el resto, y otras que funcionan como puentes entre grupos que no se comunican mucho.

Entones nos surgen dos preguntas:
1. cual es la persona mas influyente de la red y a que departamento pertenece?
2. que tan resiliente es la red? es decir, que tanto se deteriora la comunicación cuando dicha persona deja de estar disponible, y cuantas personas habrai que sacar para fragmentar la red?

El termino influyente se puede interpretar de varias maneras asi que se tomarán 3: la actividad, la intermediación y la influencia indirecta:
- **actividad:** a cuantas personas contacta o por cuantas es contactada
- **Intermediación:** en cuantas comunicaciónes funge como puente
- **Influencia indirecta:** que tan importante es quien la contacta

se busca a la persona que ocupe la posición mas alta en las tres medidas y se analiza si hay coincidencia.

Y para saber que tan reciliente es la red se eliminarán personas de la red: en orden de importancia y de manera aleatoria, y se medirá el efecto sobre la capacidad de comunicación: tamaño de la componente fuertemente conexa más grande, número de pares de personas que dejan de poder comunicarse y variación de la distancia promedio entre ellas.

## 1.2 Equivalencia matemática

## 1.2 Equivalencia matemática

### Modelo

La red de correos se modela como un grafo dirigido y sin pesos $G = (V, E)$, donde:

- $V$ es el conjunto de personas, con $n = |V| = 1\,005$.
- $E \subseteq V \times V$ es el conjunto de aristas: $(u, v) \in E$ si y solo si la persona $u$ envió al menos un correo a la persona $v$. Se tiene $m = |E| = 25\,571$ y se excluyen los auto-bucles.

Un **camino** de $u$ a $v$ es una sucesión de aristas que lleva de $u$ a $v$ respetando su dirección. La **distancia** $\operatorname{dist}(u,v)$ es el menor número de aristas de un camino de $u$ a $v$. Como el grafo no tiene pesos, esta distancia puede calcularse con una búsqueda en anchura (BFS), sin necesidad de algoritmos para grafos ponderados como Dijkstra.

### Correspondencia entre el problema real y el problema matemático

| Concepto del problema | Objeto matemático |
|---|---|
| Persona que envía o recibe muchos correos (*actividad*) | Grado de salida y de entrada de un vértice |
| Persona que actúa como puente (*intermediación*) | Centralidad de intermediación (*betweenness*) |
| Importancia de quien contacta a la persona (*influencia indirecta*) | PageRank |
| La persona deja de estar disponible | Eliminación de un vértice: grafo $G - v$ |
| La comunicación se deteriora | Cambio en las componentes fuertemente conexas, en los pares alcanzables y en las distancias |

### Actividad: centralidad de grado

El **grado de salida** $d^+(v) = |\{w \in V : (v,w) \in E\}|$ cuenta a cuántas personas escribió $v$, y el **grado de entrada** $d^-(v) = |\{u \in V : (u,v) \in E\}|$ cuenta cuántas personas le escribieron. Ambos se dividen entre $n-1$ para obtener valores comparables entre redes.

### Intermediación: centralidad de intermediación

Sea $\sigma_{st}$ el número de caminos más cortos de $s$ a $t$, y $\sigma_{st}(v)$ cuántos de ellos pasan por $v$. La centralidad de intermediación de $v$ es:

$$
C_B(v) = \sum_{\substack{s,t \in V \\ s \neq v \neq t,\ \sigma_{st} > 0}} \frac{\sigma_{st}(v)}{\sigma_{st}}
$$

Un valor alto indica que muchas comunicaciones de mínima longitud entre otras personas dependen de $v$. Para el grafo dirigido, se normaliza dividiendo entre $(n-1)(n-2)$.

### Influencia indirecta: PageRank

PageRank asigna a cada persona la probabilidad de encontrarla en el estado estacionario de un recorrido aleatorio sobre el grafo: desde la persona actual se sigue, con probabilidad $\alpha$, un correo hacia alguno de sus contactos elegido al azar y, con probabilidad $1-\alpha$, se salta a una persona cualquiera. Formalmente, $PR$ es la solución de:

$$
PR(v) = \frac{1-\alpha}{n} + \alpha \left( \sum_{u : (u,v) \in E} \frac{PR(u)}{d^+(u)} + \frac{1}{n}\sum_{u : d^+(u)=0} PR(u) \right)
$$

con $\alpha = 0.85$. El último término reparte uniformemente el peso de las personas que no enviaron correos. Una persona tiene PageRank alto no solo si recibe muchos correos, sino si los recibe de personas que a su vez son muy contactadas, lo cual formaliza la idea de "importancia de quien la contacta".

### Combinación de las tres medidas

Cada medida produce un ordenamiento distinto de las personas. Se compararán estos ordenamientos y se identificará a la persona con mejor posición conjunta, es decir, la que aparezca entre las $k$ primeras de las tres medidas o la de menor posición promedio. También se medirá el grado de coincidencia entre los ordenamientos con el coeficiente de correlación de Spearman.

### Resiliencia: eliminación de vértices

Para un conjunto $S \subset V$ de personas que dejan de estar disponibles, la red restante es el subgrafo inducido $G_S = G - S$, con vértices $V \setminus S$ y las aristas de $E$ que unen dos vértices de $V \setminus S$. Se consideran dos estrategias para elegir $S_k$, el conjunto de las $k$ personas eliminadas:

- **Dirigida:** las $k$ personas con mayor importancia según una de las medidas anteriores (ordenamiento calculado sobre $G$).
- **Aleatoria:** $k$ personas elegidas al azar, repitiendo el experimento varias veces y promediando.

En cada paso se miden tres cantidades sobre $G_S$, con $n_S = n - |S|$:

1. **Tamaño de la componente fuertemente conexa más grande,** relativo al original:
$$s(S) = \frac{|C_{\max}(G_S)|}{|C_{\max}(G)|}$$
2. **Fracción de pares que dejan de poder comunicarse.** Sea $R_S(u)$ el conjunto de nodos alcanzables desde $u$ en $G_S$:
$$f(S) = 1 - \frac{|\{(u,v) : u \neq v,\ v \in R_S(u)\}|}{n_S(n_S - 1)}$$
3. **Distancia promedio entre pares alcanzables:**
$$L(S) = \frac{1}{|P_S|} \sum_{(u,v) \in P_S} \operatorname{dist}(u,v), \quad P_S = \{(u,v) : u \neq v,\ v \in R_S(u)\}$$

**Criterio de fragmentación.** Se dice que la red queda fragmentada cuando la componente fuertemente conexa más grande se reduce a la mitad o menos de su tamaño original. El número de personas que hay que retirar es entonces:
$$k^* = \min \{ k : s(S_k) \leq 0.5 \}$$

*(Explicar la equivalencia con el problema matemático: centralidad en grafos — grado, intermediación (betweenness) — y resiliencia de redes. Definir formalmente qué es la centralidad de intermediación sin usar Dijkstra, usando BFS sobre el grafo no ponderado.)*

## 1.3 Alternativas de solución

Contrastar las siguientes opciones y justificar la selección:

| Algoritmo | Descripción | Complejidad | Observación |
|-----------|-------------|-------------|-------------|
| Centralidad de grado | Contar in/out-degree | O(V + E) | Simple, no captura intermediarios |
| Centralidad de intermediación (BFS) | Brandes con BFS | O(V · E) | Identifica puentes de comunicación |
| PageRank (implementación propia) | Flujo de autoridad | O(k · (V + E)) | Captura influencia indirecta |

*(Seleccionar y justificar. Recordar: Dijkstra no permitido.)*

## 1.4 Implementación

In [ ]:
# ── Tarea 1: Resiliencia y nodo más influyente ────────────────────────────────
# TODO: implementar el algoritmo seleccionado
# Sugerencia de estructura:
#
#   def centralidad_grado(grafo, grafo_inv):
#       """Retorna dict {nodo: (grado_entrada, grado_salida, total)}"""
#       ...
#
#   def bfs_caminos(grafo, origen):
#       """BFS desde 'origen'; retorna distancias y listas de predecesores."""
#       ...
#
#   def centralidad_intermediacion(grafo, nodos):
#       """Algoritmo de Brandes usando BFS (sin Dijkstra)."""
#       ...
#
#   def impacto_eliminacion(grafo, grafo_inv, nodo_critico):
#       """Simula la eliminación de 'nodo_critico' y mide la fragmentación."""
#       ...

pass

## 1.5 Resultados y análisis

*(Presentar: el top-k de nodos más influyentes, la visualización de centralidades, y el análisis del impacto de eliminar el nodo más importante. ¿Cuánto se fragmenta la red?)*

In [ ]:
# ── Visualización de resultados — Tarea 1 ────────────────────────────────────
# TODO

---
# Tarea 2 — Comunicación inter-departamental
**Integrante:** *(Nombre integrante 2)*

## 2.1 Descripción del problema

*(Describir el problema real: ¿cómo se comunican entre sí los 42 departamentos del instituto? ¿Hay departamentos que funcionan como silos aislados? ¿Cuáles son los puentes departamentales?)*

## 2.2 Equivalencia matemática

*(Explicar la construcción de un grafo de departamentos a partir del grafo original: contracción de nodos por comunidad etiquetada. El grafo resultante tendrá a lo sumo 42 nodos; cada arista pesada representa el volumen de correos entre dos departamentos.)*

## 2.3 Alternativas de solución

| Algoritmo | Descripción | Complejidad | Observación |
|-----------|-------------|-------------|-------------|
| Grafo de departamentos (contracción) | Construir meta-grafo | O(E) | Directo con las etiquetas |
| Matriz de flujo inter-dept | Conteo de aristas cruzadas | O(E) | Visualizable como heatmap |
| BFS/DFS sobre el meta-grafo | Accesibilidad entre departamentos | O(42²) | Sobre el grafo condensado |

*(Seleccionar y justificar.)*

## 2.4 Implementación

In [ ]:
# ── Tarea 2: Comunicación inter-departamental ─────────────────────────────────
# TODO: implementar el algoritmo seleccionado
# Sugerencia de estructura:
#
#   def construir_grafo_departamentos(grafo, departamento):
#       """Retorna meta-grafo: {dept_u: {dept_v: num_correos}}"""
#       ...
#
#   def departamentos_mas_activos(meta_grafo):
#       """Top-k departamentos por volumen total de correos enviados/recibidos."""
#       ...
#
#   def departamentos_aislados(meta_grafo, todos_los_depts):
#       """Departamentos con pocas o ninguna conexión externa."""
#       ...

pass

## 2.5 Resultados y análisis

*(Presentar: heatmap del flujo inter-departamental, top departamentos más conectados, departamentos más aislados, y el meta-grafo visualizado.)*

In [ ]:
# ── Visualización de resultados — Tarea 2 ────────────────────────────────────
# TODO

---
# Tarea 3 — Componentes conexas y accesibilidad de la información
**Integrante:** *Angel Camilo Pachon Padilla*

## 3.1 Descripción del problema

En una red de correos, no todos se comunican con todos. La pregunta es: si alguien envía un mensaje, ¿hasta dónde llega realmente? ¿Existen grupos que nunca reciben correos del exterior? Y al revés: ¿hay personas desde las que se puede llegar a cualquiera? La idea es mapear la estructura de alcanzabilidad.

Describir el problema real: *si hay un rumor, un comunicado o un archivo empieza en un nodo determinado, ¿a cuántos miembros de la institución puede llegar? ¿Existen grupos completamente incomunicados? ¿Hay personas que pueden comunicarse con todos?*

## 3.2 Equivalencia matemática

Dado un grafo dirigido $G = (V, E)$, donde $V$ es el conjunto de nodos y $E$ el de aristas, se definen dos tipos de componentes conexas.

Una **componente débilmente conexa (CDbC)** agrupa nodos que están conectados si se ignora la dirección de las aristas, es decir, tratando $G$ como un grafo no dirigido. Formalmente, $u$ y $v$ pertenecen a la misma CDbC si existe algún camino entre ellos en $G^{\text{no dirigido}}$.

Una **componente fuertemente conexa (CFC)** es más restrictiva: agrupa nodos que se pueden alcanzar **mutuamente** siguiendo las aristas en su dirección original. $u$ y $v$ están en la misma CFC si y solo si existe un camino $u \rightsquigarrow v$ **y** un camino $v \rightsquigarrow u$ en $G$.

La **alcanzabilidad** de un nodo $u$ se define como el conjunto:

$$
R(u) = \{v \in V \mid \text{existe un camino } u \rightsquigarrow v \text{ en } G\}
$$

El tamaño de $R(u)$ indica cuántas personas pueden recibir información que salga desde $u$. Un nodo con $|R(u)| = 1$ solo se alcanza a sí mismo; un nodo en la CFC gigante tiene $|R(u)|$ cercano a $|V|$.

El **diámetro** de la red se define como la mayor distancia entre cualquier par de nodos alcanzables entre sí:

$$
d = \max_{\substack{u,v \in V \\ v \in R(u)}} \operatorname{dist}(u,v)
$$

## 3.3 Alternativas de solución

Los algoritmos considerados son:

- **BFS** (Búsqueda en Anchura, *Breadth-First Search*): recorre el grafo nivel por nivel desde un nodo origen, visitando primero los vecinos más cercanos. Útil para medir alcanzabilidad y distancias mínimas.
- **DFS** (Búsqueda en Profundidad, *Depth-First Search*): recorre el grafo siguiendo un camino hasta el fondo antes de retroceder. Base de los algoritmos de componentes conexas.
- **Kosaraju**: algoritmo que encuentra todas las componentes fuertemente conexas mediante dos DFS — una sobre el grafo original y otra sobre el grafo transpuesto.
- **Tarjan**: algoritmo que encuentra las mismas componentes fuertemente conexas pero en una sola DFS, usando una pila y valores auxiliares llamados `low-link`.

Para encontrar todas las componentes fuertemente conexas se escoge **Kosaraju**, complementado con **BFS** para los ejemplos de alcanzabilidad puntual.

Kosaraju opera en dos fases sobre el grafo $G = (V, E)$:

1. Una primera DFS sobre $G$ que registra el **orden de finalización** de cada nodo.
2. Una segunda DFS sobre el **grafo transpuesto** $G^T$ — donde todas las aristas están invertidas — procesando los nodos en orden inverso de finalización. Cada árbol generado en esta segunda pasada es una componente fuertemente conexa.

Se elige Kosaraju sobre Tarjan porque, aunque ambos tienen complejidad $O(V + E)$, la lógica es más directa: dos recorridos independientes sin estructuras auxiliares de valores `low-link`. Además, $G^T$ ya está construido desde la carga inicial del dataset (variable `grafo_inv`), por lo que no representa trabajo adicional.

## 3.4 Implementación

In [ ]:
# ── Tarea 3: Componentes conexas y accesibilidad ──────────────────────────────

def bfs_alcanzabilidad(grafo, origen):
    """Retorna el conjunto de nodos alcanzables desde 'origen' usando BFS."""
    visitados = {origen}
    cola = deque([origen])
    while cola:
        u = cola.popleft()
        for v in grafo[u]:
            if v not in visitados:
                visitados.add(v)
                cola.append(v)
    return visitados


def kosaraju(grafo, grafo_inv, nodos):
    """
    Algoritmo de Kosaraju — retorna lista de CFC (cada CFC es un set de nodos).
    Usa DFS iterativo para evitar límites de recursión en grafos grandes.
    """
    # Paso 1: DFS sobre el grafo original → registrar orden de finalización
    visitados = set()
    orden_fin = []

    for inicio in nodos:
        if inicio in visitados:
            continue
        pila = [(inicio, iter(grafo[inicio]))]
        visitados.add(inicio)
        while pila:
            nodo, vecinos = pila[-1]
            try:
                v = next(vecinos)
                if v not in visitados:
                    visitados.add(v)
                    pila.append((v, iter(grafo[v])))
            except StopIteration:
                pila.pop()
                orden_fin.append(nodo)

    # Paso 2: DFS sobre el grafo transpuesto en orden inverso de finalización
    # Cada árbol resultante es una componente fuertemente conexa
    visitados2 = set()
    cfcs = []

    for inicio in reversed(orden_fin):
        if inicio in visitados2:
            continue
        componente = []
        pila = [inicio]
        visitados2.add(inicio)
        while pila:
            nodo = pila.pop()
            componente.append(nodo)
            for v in grafo_inv[nodo]:
                if v not in visitados2:
                    visitados2.add(v)
                    pila.append(v)
        cfcs.append(set(componente))

    return cfcs


def wcc(grafo, grafo_inv, nodos):
    """
    Componentes débilmente conexas — trata el grafo como no dirigido
    usando BFS sobre la unión de grafo y grafo transpuesto.
    """
    visitados = set()
    componentes = []

    for inicio in nodos:
        if inicio in visitados:
            continue
        comp = set()
        cola = deque([inicio])
        visitados.add(inicio)
        while cola:
            u = cola.popleft()
            comp.add(u)
            for v in grafo[u] | grafo_inv[u]:
                if v not in visitados:
                    visitados.add(v)
                    cola.append(v)
        componentes.append(comp)

    return componentes

## 3.5 Resultados y análisis


**Componentes débilmente conexas (CDbC):** el algoritmo encontró una única CDbC que agrupa los 986 nodos del grafo. Esto significa que, ignorando la dirección de los correos, toda la institución forma una sola red conectada: no existe ningún grupo completamente aislado del resto.

**Componentes fuertemente conexas (CFC):** el panorama cambia radicalmente al considerar la dirección. Kosaraju identificó 184 CFCs en total:

| Componente | Tamaño | % del grafo |
|---|---|---|
| CFC gigante | 803 nodos | 81.4 % |
| CFCs periféricas | 183 componentes | 18.6 % |

Las 183 CFCs periféricas tienen exactamente tamaño 1, es decir, son nodos que no forman un ciclo de comunicación con nadie: o solo envían correos sin recibir respuesta, o solo reciben sin que nadie les conteste de vuelta.

**Ejemplos de alcanzabilidad (BFS):**

- El nodo 0 (departamento 1), que pertenece a la CFC gigante, puede alcanzar **965 nodos (97.9 %)** de la red. Desde ahí un mensaje llegaría prácticamente a toda la institución.
- El nodo 1 (departamento 1), un nodo periférico, solo se alcanza **a sí mismo (0.1 %)**. Sus correos no llegan a ningún otro nodo, o nadie le responde.

La distribución de tamaños confirma visualmente la estructura típica de redes de comunicación reales: una componente gigante dominante y una larga cola de nodos marginales. La CDbC única y la CFC gigante juntas sugieren que la institución está bien conectada a nivel de infraestructura, pero la comunicación *recíproca* está concentrada en un núcleo activo del 81.4 % de los miembros.


In [ ]:
# ── Ejecutar algoritmos ────────────────────────────────────────────────────────
import time

t0 = time.time()
cfcs = kosaraju(grafo, grafo_inv, nodos)
t1 = time.time()
wccs = wcc(grafo, grafo_inv, nodos)
t2 = time.time()

cfcs_ord = sorted(cfcs, key=len, reverse=True)
wccs_ord = sorted(wccs, key=len, reverse=True)

print('=== Componentes Fuertemente Conexas (CFC) ===')
print(f'Total de CFCs:         {len(cfcs)}')
print(f'CFC gigante:           {len(cfcs_ord[0])} nodos  ({100*len(cfcs_ord[0])/num_nodos:.1f} %)')
print(f'CFCs de tamaño 1:      {sum(1 for c in cfcs if len(c) == 1)}')
print(f'Tiempo Kosaraju:       {t1-t0:.3f} s')
print()
print('=== Componentes Débilmente Conexas (CDbC) ===')
print(f'Total de CDbCs:        {len(wccs)}')
print(f'CDbC gigante:          {len(wccs_ord[0])} nodos  ({100*len(wccs_ord[0])/num_nodos:.1f} %)')
print(f'Tiempo WCC:            {t2-t1:.3f} s')

# ── Distribución de tamaños ────────────────────────────────────────────────────
tamaños_cfc = [len(c) for c in cfcs]
tamaños_wcc = [len(c) for c in wccs]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(tamaños_cfc, bins=40, color='steelblue', edgecolor='white', log=True)
axes[0].set_title('Distribución de tamaños — CFC')
axes[0].set_xlabel('Tamaño de la componente')
axes[0].set_ylabel('Frecuencia (escala log)')

axes[1].hist(tamaños_wcc, bins=20, color='darkorange', edgecolor='white', log=True)
axes[1].set_title('Distribución de tamaños — CDbC')
axes[1].set_xlabel('Tamaño de la componente')

plt.tight_layout()
plt.show()

# ── Ejemplos de alcanzabilidad (BFS) ──────────────────────────────────────────
nodo_cfc_gigante = next(iter(cfcs_ord[0]))    # nodo dentro de la CFC gigante
nodo_periferico  = next(iter(cfcs_ord[-1]))   # nodo en una CFC de tamaño 1

print('\n=== Ejemplos de alcanzabilidad (BFS) ===')
for origen in [nodo_cfc_gigante, nodo_periferico]:
    alcanzables = bfs_alcanzabilidad(grafo, origen)
    dept = departamento.get(origen, '?')
    print(f'Nodo {origen:>4}  (dept {dept:>2}): '
          f'alcanza {len(alcanzables):>4} nodos  ({100*len(alcanzables)/num_nodos:.1f} %)')

=== Componentes Fuertemente Conexas (CFC) ===
Total de CFCs:         0


IndexError: list index out of range

---
# Tarea 4 — Detección de comunidades y comparación con departamentos reales
**Integrante:** *(Samuel Alejandro Fuentes Sandoval)*

## 4.1 Descripción del problema

*(Describir el problema real: ¿la gente se agrupa en sus correos según su departamento oficial, o las comunidades de comunicación cruzan fronteras departamentales? ¿Puede detectarse la estructura departamental solo mirando el grafo, sin las etiquetas?)*

## 4.2 Equivalencia matemática

*(Explicar la detección de comunidades como partición del grafo que maximiza la cohesión interna y minimiza las aristas entre grupos. Definir modularidad Q y el concepto de corte mínimo entre comunidades. Relacionar con la comparación entre partición detectada y la partición real por departamentos.)*

## 4.3 Alternativas de solución

| Algoritmo | Descripción | Complejidad | Observación |
|-----------|-------------|-------------|-------------|
| Label Propagation | Propaga etiquetas por vecindad | O(k · (V + E)) | Rápido, no determinista |
| Girvan-Newman | Elimina aristas de alta intermediación | O(V · E²) | Lento, pero interpretable |
| Louvain (implementación propia) | Optimización de modularidad | O(V · log V) | Eficiente, jerárquico |

*(Seleccionar y justificar. Para comparar el resultado con las etiquetas reales, usar el Índice de Rand ajustado o la Pureza, implementados a mano.)*

## 4.4 Implementación

In [ ]:
# ── Tarea 4: Detección de comunidades ─────────────────────────────────────────
# TODO: implementar el algoritmo seleccionado
# Sugerencia de estructura:
#
#   def label_propagation(grafo, nodos, semilla=42):
#       """Asigna una comunidad a cada nodo propagando etiquetas iterativamente.
#          Retorna dict {nodo: id_comunidad}."""
#       ...
#
#   def pureza(comunidades_detectadas, departamento_real):
#       """Mide cuánto coincide la partición detectada con la real.
#          Retorna un valor en [0, 1] (1 = coincidencia perfecta)."""
#       ...
#
#   def modularidad(grafo, comunidades, num_aristas):
#       """Calcula la modularidad Q de una partición dada."""
#       ...

pass

In [ ]:
import random
from collections import Counter

def label_propagation(grafo, nodos, max_iter=20, semilla=42):
    """
    Algoritmo de Propagación de Etiquetas para detectar comunidades.
    Retorna un diccionario {nodo: id_comunidad}.
    """
    random.seed(semilla)
    # 1. Inicialización: cada nodo comienza con una etiqueta única (su propio ID)
    comunidades = {nodo: nodo for nodo in nodos}

    # Convertir el grafo dirigido en no-dirigido implícitamente para una mejor propagación
    # Creando una lista de vecinos considerando enlaces de envío y recepción
    vecinos_nd = defaultdict(set)
    for u in grafo:
        for v in grafo[u]:
            vecinos_nd[u].add(v)
            vecinos_nd[v].add(u)

    lista_nodos = list(nodos)

    # 2. Iteración
    for iteracion in range(max_iter):
        random.shuffle(lista_nodos) # Evitar sesgos de orden
        cambios = 0

        for u in lista_nodos:
            vecinos = vecinos_nd[u]
            if not vecinos:
                continue

            # Contar la frecuencia de las etiquetas de los vecinos
            etiquetas_vecinos = [comunidades[v] for v in vecinos]
            conteo = Counter(etiquetas_vecinos)

            # Encontrar la(s) etiqueta(s) más común(es)
            max_frecuencia = max(conteo.values())
            etiquetas_mas_comunes = [lbl for lbl, frec in conteo.items() if frec == max_frecuencia]

            # Elegir una etiqueta ganadora al azar para romper empates
            etiqueta_elegida = random.choice(etiquetas_mas_comunes)

            if comunidades[u] != etiqueta_elegida:
                comunidades[u] = etiqueta_elegida
                cambios += 1

        # Si no hay cambios de etiquetas, el algoritmo ha convergido
        if cambios == 0:
            print(f'-> LPA convergió en la iteración {iteracion + 1}')
            break
    else:
        print('-> LPA alcanzó el límite de iteraciones sin converger totalmente')

    return comunidades

In [ ]:
def calcular_pureza(comunidades_detectadas, departamento_real):
    """
    Calcula la pureza de la partición detectada en comparación con las etiquetas reales.
    Retorna un valor float entre 0.0 y 1.0 (donde 1.0 es coincidencia ideal).
    """
    # Agrupar nodos por la comunidad detectada
    grupos_detectados = defaultdict(list)
    for nodo, com_id in comunidades_detectadas.items():
        grupos_detectados[com_id].append(nodo)

    intersecciones_maximas = 0
    total_nodos = len(comunidades_detectadas)

    for com_id, miembros in grupos_detectados.items():
        # Contar qué departamento oficial predomina en esta comunidad
        conteos_dept = Counter(departamento_real[nodo] for nodo in miembros if nodo in departamento_real)
        if conteos_dept:
            clase_mayoritaria = max(conteos_dept.values())
            intersecciones_maximas += clase_mayoritaria

    return intersecciones_maximas / total_nodos

def calcular_modularidad_dirigida(grafo, comunidades_detectadas, num_aristas):
    """
    Calcula la modularidad Q para un grafo dirigido dada una partición de comunidades.
    """
    m = num_aristas
    if m == 0:
        return 0.0

    # Calcular grado de salida (out-degree) y entrada (in-degree) para cada nodo
    k_out = {u: len(grafo[u]) for u in grafo}

    k_in = defaultdict(int)
    for u in grafo:
        for v in grafo[u]:
            k_in[v] += 1

    Q = 0.0
    # Doble sumatoria sobre los nodos que comparten la misma comunidad
    # Para optimizar, agrupamos primero los nodos por comunidad
    grupos = defaultdict(list)
    for nodo, com in comunidades_detectadas.items():
        grupos[com].append(nodo)

    for com, miembros in grupos.items():
        for u in miembros:
            for v in miembros:
                # Si hay arista dirigida de u a v, la variable indicadora A_{uv} = 1, si no 0
                A_uv = 1.0 if v in grafo[u] else 0.0
                prob_aleatoria = (k_out[u] * k_in[v]) / m
                Q += (A_uv - prob_aleatoria)

    return Q / m

## 4.5 Resultados y análisis

*(Presentar: comunidades detectadas, valor de modularidad, comparación con los departamentos reales (pureza o Rand), y reflexión sobre hasta qué punto la estructura de correos refleja la estructura organizacional.)*

In [ ]:
# ── Visualización de resultados — Tarea 4 ────────────────────────────────────
# TODO

In [ ]:
# Executing Tarea 4 Algorithms and Metrics
print("Ejecutando Detección de Comunidades (Label Propagation)...")
comunidades_lpa = label_propagation(grafo, nodos, semilla=112)

# Obtener número de comunidades y distribución de tamaños
com_dist = Counter(comunidades_lpa.values())

# Calcular métricas
pureza_lpa = calcular_pureza(comunidades_lpa, departamento)
modularidad_lpa = calcular_modularidad_dirigida(grafo, comunidades_lpa, num_aristas)

print(f"\n=== RESULTADOS TAREA 4 ===")
print(f"Número de comunidades detectadas: {len(com_dist)}")
print(f"Modularidad Q del grafo:          {modularidad_lpa:.4f}")
print(f"Pureza (frente a deptos reales):  {pureza_lpa:.4f}  ({pureza_lpa * 100:.1f}% de coincidencia)")

# Mostrar tamaños de las 5 comunidades más grandes
print("\nTamaño de las 5 comunidades más grandes:")
for idx, (com_id, tam) in enumerate(com_dist.most_common(5)):
    print(f"  Comunidad {idx+1} (ID: {com_id}): {tam} nodos")

---
# 3. Conclusiones generales

*(Sintetizar los hallazgos más relevantes de las cuatro tareas. Relacionar los resultados entre sí: por ejemplo, ¿coincide el nodo más influyente (Tarea 1) con un nodo que pertenezca a una comunidad grande (Tarea 4)? ¿El departamento más conectado (Tarea 2) es el que tiene más nodos en la SCC principal (Tarea 3)?)*

- La red forma una **única componente débilmente conexa** (986 nodos): ignorando la dirección de los correos, nadie está completamente aislado del resto de la institución.

- Al considerar la dirección, solo el **81.4% de los nodos (803)** pertenecen a la componente fuertemente conexa gigante. *El núcleo donde la información puede circular en cualquier sentido.*

- Los **183 nodos restantes** son completamente periféricos: cada uno forma su propia componente de tamaño 1, es decir, no mantiene comunicación recíproca con nadie.

- Desde un nodo dentro del núcleo activo es posible alcanzar hasta el **97.9% de la red** con un simple BFS; desde un nodo periférico, solo se llega a sí mismo.

- La institución está bien conectada a nivel de infraestructura, pero la **comunicación recíproca está concentrada** en un subconjunto activo de sus miembros.

---
# 4. Referencias

- Yin, H., Benson, A. R., Leskovec, J., y Gleich, D. F. (2017). *Local Higher-order Graph Clustering*. KDD.
- Leskovec, J., Kleinberg, J., y Faloutsos, C. (2007). *Graph Evolution: Densification and Shrinking Diameters*. ACM TKDD, 1(1).
- Cormen, T. H. et al. (2022). *Introduction to Algorithms*, 4th ed. MIT Press. (Capítulos 20–22: grafos, BFS, DFS, SCC.)